In [44]:
# Sqrt

import numpy as np

data = np.load("/Users/isaaclee/Wildfire_Research/data/training_testing_data_for_predicting_fire_area_from_ign_time_conditions.npy")
x = data[:, :1] * 50000
sqrt_x = np.sqrt(x)

x_max = sqrt_x.max()
x_norm = sqrt_x / x_max
print(x_max)

data[:, :1] = x_norm

np.save("/Users/isaaclee/Wildfire_Research/data/training_testing_data_for_predicting_fire_area_from_ign_time_conditions_sqrt.npy", data)
np.savez("/Users/isaaclee/Wildfire_Research/data/sqrt_x_scaling_factor.npz", x_max=x_max)



199.83784432745998


## Regular Log

In [45]:
import numpy as np

data = np.load("/Users/isaaclee/Wildfire_Research/data/training_testing_data_for_predicting_fire_area_from_ign_time_conditions.npy")

x = data[:, :1] * 50000
x_log = np.log(x)

x_max = x_log.max()
x_norm = x_log / x_max
print(x_max)

data[:, :1] = x_norm

np.save("/Users/isaaclee/Wildfire_Research/data/training_testing_data_for_predicting_fire_area_from_ign_time_conditions_ln.npy", data)
np.savez("/Users/isaaclee/Wildfire_Research/data/x_scaling_factor.npz", x_max=x_max)


10.595012518653586


In [46]:
import numpy as np

real_fire_data = np.load('/Users/isaaclee/Wildfire_Research/data/Barnes_Eaton_Oak_Palisades_fire_area_pred_from_IC_test_cases_fixed.npy')

x = real_fire_data[:, :1] * 50000
x_log = np.log(x)

x_max = x_log.max()
x_norm = x_log / x_max
print(x_max)

real_fire_data[:, :1] = x_norm
np.save("/Users/isaaclee/Wildfire_Research/data/real_fire_data_ln.npy", real_fire_data)
np.savez("/Users/isaaclee/Wildfire_Research/data/real_x_scaling_factor.npz", x_max=x_max)


9.781929845462189


## Shifted Log

In [47]:
# log (constant * max fire area + 1) = 1 (solve for constant)

import numpy as np

data = np.load("/Users/isaaclee/Wildfire_Research/data/training_testing_data_for_predicting_fire_area_from_ign_time_conditions.npy")

x = data[:, :1] * 50000
max_x = np.max(x)

constant = (np.e - 1) / max_x

new_x = np.log(constant * x + 1)
data[:, :1] = new_x
print(new_x.min())
print(new_x.max())

np.save("/Users/isaaclee/Wildfire_Research/data/training_testing_data_ln_shifted.npy", data)
np.savez("/Users/isaaclee/Wildfire_Research/data/x_scaling_constant.npz", constant=constant)


print(data.shape)


6.645072180753334e-06
1.0
(15200, 24)


In [48]:
import numpy as np

real_fire_data = np.load('/Users/isaaclee/Wildfire_Research/data/Barnes_Eaton_Oak_Palisades_fire_area_pred_from_IC_test_cases_fixed.npy')

x = real_fire_data[:, :1] * 50000
max_x = np.max(x)

constant = (np.e - 1) / max_x
new_x = np.log(constant * x + 1)
real_fire_data[:, :1] = new_x

print(new_x.min())
print(new_x.max())

print(real_fire_data.shape)

np.save("/Users/isaaclee/Wildfire_Research/data/real_fire_data_ln_shifted.npy", real_fire_data)
np.savez("/Users/isaaclee/Wildfire_Research/data/real_x_scaling_constant.npz", constant=constant)


0.251146346250832
1.0
(8, 24)


## Recursive transformation

In [49]:
import numpy as np
import scipy.io
from scipy.stats import norm

mat_train_data = scipy.io.loadmat('/Users/isaaclee/Wildfire_Research/data/training_data_for_predicting_scalar_fire_area_recursively.mat')
train_data = mat_train_data['training_data']

mat_test_data = scipy.io.loadmat('/Users/isaaclee/Wildfire_Research/data/testing_data_for_predicting_scalar_fire_area_recursively.mat')
test_data = mat_test_data['testing_data']

# First normalization by max
x1_max = max(train_data[:,0])
x2_max = max(train_data[:,1])
x3_max = max(train_data[:,2])

train_data[:,0] = train_data[:,0]/x1_max
train_data[:,1] = train_data[:,1]/x2_max
train_data[:,2] = train_data[:,2]/x3_max

test_data[:,0] = test_data[:,0]/x1_max
test_data[:,1] = test_data[:,1]/x2_max
test_data[:,2] = test_data[:,2]/x3_max

# make sure there are not too many values above 1 for test

# Clippings for CDF
eps = 1e-8
train_data[:,0] = np.clip(train_data[:,0], eps, 1-eps)
train_data[:,1] = np.clip(train_data[:,1], eps, 1-eps)
train_data[:,2] = np.clip(train_data[:,2], eps, 1-eps)

test_data[:,0] = np.clip(test_data[:,0], eps, 1-eps)
test_data[:,1] = np.clip(test_data[:,1], eps, 1-eps)
test_data[:,2] = np.clip(test_data[:,2], eps, 1-eps)

# Inverse CDF
train_data[:,0] = norm.ppf(train_data[:,0])
train_data[:,1] = norm.ppf(train_data[:,1])
train_data[:,2] = norm.ppf(train_data[:,2])

test_data[:,0] = norm.ppf(test_data[:,0])
test_data[:,1] = norm.ppf(test_data[:,1])
test_data[:,2] = norm.ppf(test_data[:,2])

# Final normalization
x1_max_new = max(train_data[:,0])
x2_max_new = max(train_data[:,1])
x3_max_new = max(train_data[:,2])

x1_min_new = min(train_data[:,0])
x2_min_new = min(train_data[:,1])
x3_min_new = min(train_data[:,2])

train_data[:,0] = (train_data[:,0] - x1_min_new)/(x1_max_new - x1_min_new)
train_data[:,1] = (train_data[:,1] - x2_min_new)/(x2_max_new - x2_min_new)
train_data[:,2] = (train_data[:,2] - x3_min_new)/(x3_max_new - x3_min_new)

test_data[:,0] = (test_data[:,0] - x1_min_new)/(x1_max_new - x1_min_new)
test_data[:,1] = (test_data[:,1] - x2_min_new)/(x2_max_new - x2_min_new)
test_data[:,2] = (test_data[:,2] - x3_min_new)/(x3_max_new - x3_min_new)

In [50]:
# Finding maximum euclidean distance

from scipy.spatial.distance import pdist

points = train_data[:,:3]
n = points.shape[0]

max_dist = np.max(pdist(points, metric='euclidean'))
print(max_dist)

np.save('/Users/isaaclee/Wildfire_Research/data/recursive_train_data_norm.npy', train_data)
np.save('/Users/isaaclee/Wildfire_Research/data/recursive_test_data_norm.npy', test_data)

np.save('/Users/isaaclee/Wildfire_Research/recursive_data/first_x1_max.npy', x1_max)
np.save('/Users/isaaclee/Wildfire_Research/recursive_data/first_x2_max.npy', x2_max)
np.save('/Users/isaaclee/Wildfire_Research/recursive_data/first_x3_max.npy', x3_max)

np.save('/Users/isaaclee/Wildfire_Research/recursive_data/recursive_x1_min.npy', x1_min_new)
np.save('/Users/isaaclee/Wildfire_Research/recursive_data/recursive_x2_min.npy', x2_min_new)
np.save('/Users/isaaclee/Wildfire_Research/recursive_data/recursive_x3_min.npy', x3_min_new)

np.save('/Users/isaaclee/Wildfire_Research/recursive_data/recursive_x1_max.npy', x1_max_new)
np.save('/Users/isaaclee/Wildfire_Research/recursive_data/recursive_x2_max.npy', x2_max_new)
np.save('/Users/isaaclee/Wildfire_Research/recursive_data/recursive_x3_max.npy', x3_max_new)

1.7320508075688772
